In [ ]:
#!pip install -U transformers datasets evaluate accelerate


In [ ]:
import os
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)


In [ ]:
SEED = 42
BASE_MODEL = "distilbert/distilbert-base-uncased"
MAX_LENGTH = 128
PUSH_TO_HUB = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)


In [ ]:
project_path = Path.cwd()
if project_path.name == "notebooks":
    project_path = project_path.parent
project_path = project_path.resolve()

datasets_dir = project_path / "datasets"
metrics_dir = project_path / "metrics"
models_dir = project_path / "models"
train_path = datasets_dir / "training_data.csv"
external_test_path = datasets_dir / "testing_data.csv"
model_output_dir = models_dir / "distilbert_baselines_New"
final_model_dir = models_dir / "distilbert_baseline_final"
prediction_path = project_path / "predictions_bert.csv"


In [ ]:
os.chdir(project_path)

# Create folder structure
models_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)
datasets_dir.mkdir(parents=True, exist_ok=True)

print(f"Project path: {project_path}")
print(f"Training data: {train_path}")
print(f"External test data: {external_test_path}")


##functions

In [ ]:
metrics_dir.mkdir(parents=True, exist_ok=True)

experiment_csv_path = metrics_dir / "model_experiments.csv"

EXPERIMENT_COLUMNS = [
    "experiment_id",
    "model_name",
    "base_model",
    "dataset_version",
    "train_size",
    "validation_size",
    "learning_rate",
    "batch_size",
    "epochs",
    "weight_decay",
    "max_length",
    "preprocessing_strategy",
    "improvement_strategy_used",
    "train_accuracy",
    "validation_accuracy",
    "validation_precision",
    "validation_recall",
    "validation_f1",
    "validation_loss",
    "notes",
]

def append_experiment_result(experiment_result, csv_path=experiment_csv_path):
    new_row = pd.DataFrame([experiment_result])

    for column in EXPERIMENT_COLUMNS:
        if column not in new_row.columns:
            new_row[column] = None

    new_row = new_row[EXPERIMENT_COLUMNS]

    if csv_path.exists():
        previous_results = pd.read_csv(csv_path)
        updated_results = pd.concat(
            [previous_results, new_row],
            ignore_index=True
        )
    else:
        updated_results = new_row

    updated_results.to_csv(csv_path, index=False)
    return updated_results


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
    }

In [ ]:
MAX_LENGTH = 128

def tokenize_dataset(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


##Data preparation

In [ ]:
data = pd.read_csv(
    train_path,
    sep="	",
    header=None,
    names=["label", "headline"],
    encoding="utf-8-sig",
)

external_test_df = pd.read_csv(
    external_test_path,
    sep="	",
    header=None,
    names=["label", "text"],
    encoding="utf-8-sig",
)


In [ ]:
data[["headline", "label"]].head()

In [ ]:
text_col = "headline"
label_col = "label"

print("Raw data shape:", data.shape)
print(data[[text_col, label_col]].head())
print(data[label_col].value_counts())
print("Missing headlines:", data[text_col].isna().sum())
print("Missing labels:", data[label_col].isna().sum())

data = data.dropna(subset=[text_col, label_col]).copy()
data[text_col] = data[text_col].astype(str).str.strip()
data = data[data[text_col] != ""].copy()
data = data.drop_duplicates(subset=[text_col]).reset_index(drop=True)
data[label_col] = data[label_col].astype(int)

print("Clean data shape:", data.shape)
print(data[label_col].value_counts())


In [ ]:
train_df, val_df = train_test_split(
    data[[text_col, label_col]],
    test_size=0.2,
    random_state=SEED,
    stratify=data[label_col],
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))


print("Train labels:")
print(train_df[label_col].value_counts(normalize=True))
print("Validation labels:")
print(val_df[label_col].value_counts(normalize=True))

train_texts = set(train_df[text_col])
val_texts = set(val_df[text_col])

overlap = train_texts.intersection(val_texts)
print("Train/validation exact overlap:", len(overlap))

In [ ]:
dataset = DatasetDict({
    "train": Dataset.from_dict({
        "text": train_df[text_col].tolist(),
        "label": train_df[label_col].tolist(),
    }),
    "validation": Dataset.from_dict({
        "text": val_df[text_col].tolist(),
        "label": val_df[label_col].tolist(),
    }),
})


##model initialization

###TOKENIZATION

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenized_dataset = dataset.map(tokenize_dataset, batched=True)


###call the model

In [ ]:
id2label = {
    0: "FAKE",
    1: "REAL",
}

label2id = {
    "FAKE": 0,
    "REAL": 1,
}

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir=str(model_output_dir),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    logging_strategy="steps",
    logging_steps=250,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    push_to_hub=False,
    seed=SEED,
)


##training

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
best_model_checkpoint = trainer.state.best_model_checkpoint
best_metric = trainer.state.best_metric

trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

print(train_result)
print("Best checkpoint:", best_model_checkpoint)
print("Best metric:", best_metric)
print(f"Saved model to: {final_model_dir}")


##evaluate

In [ ]:
model_path = final_model_dir

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

print("Model loaded")


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [ ]:
validation_metrics = trainer.evaluate(
    eval_dataset=tokenized_dataset["validation"],
    metric_key_prefix="validation"
)
print(validation_metrics)

validation_output = trainer.predict(
    tokenized_dataset["validation"],
    metric_key_prefix="validation",
)
validation_logits = validation_output.predictions
validation_labels = validation_output.label_ids
validation_preds = validation_logits.argmax(axis=1)

report_df = pd.DataFrame(
    classification_report(
        validation_labels,
        validation_preds,
        target_names=["FAKE", "REAL"],
        output_dict=True,
        zero_division=0,
    )
).transpose()

report_df.to_csv(metrics_dir / "classification_report_validation.csv")

cm = confusion_matrix(validation_labels, validation_preds)
cm_df = pd.DataFrame(
    cm,
    index=["actual_FAKE", "actual_REAL"],
    columns=["predicted_FAKE", "predicted_REAL"],
)
cm_df.to_csv(metrics_dir / "confusion_matrix_validation.csv")


In [ ]:
print(model_path)
print(model.config.id2label)
print(model.config.label2id)


In [ ]:
print("Best checkpoint:", globals().get("best_model_checkpoint"))
print("Best metric:", globals().get("best_metric"))


In [ ]:
experiment_result = {
    "experiment_id": datetime.now().strftime("%Y%m%d_%H%M%S"),
    "model_name": "distilbert_baseline_v1",
    "base_model": BASE_MODEL,
    "dataset_version": "training_data.csv",
    "train_size": len(tokenized_dataset["train"]),
    "validation_size": len(tokenized_dataset["validation"]),
    "learning_rate": training_args.learning_rate,
    "batch_size": training_args.per_device_train_batch_size,
    "epochs": training_args.num_train_epochs,
    "weight_decay": training_args.weight_decay,
    "max_length": MAX_LENGTH,
    "preprocessing_strategy": "strip text, drop missing values, drop duplicate headlines",
    "improvement_strategy_used": "clean DistilBERT baseline",
    "train_accuracy": None,
    "validation_accuracy": validation_metrics.get("validation_accuracy"),
    "validation_precision": validation_metrics.get("validation_precision"),
    "validation_recall": validation_metrics.get("validation_recall"),
    "validation_f1": validation_metrics.get("validation_f1"),
    "validation_loss": validation_metrics.get("validation_loss"),
    "notes": "Clean run from restarted runtime. Hugging Face Hub push disabled by default.",
}

all_results = append_experiment_result(experiment_result)
all_results.tail()


##predict

In [ ]:
model_path = final_model_dir

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

print("Model loaded")


In [ ]:
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

In [ ]:
external_texts = external_test_df["text"].astype(str).tolist()

predictions = classifier(
    external_texts,
    batch_size=32,
    truncation=True,
    max_length=MAX_LENGTH,
)

label_mapping = {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "FAKE": 0,
    "REAL": 1,
}

external_predictions_df = external_test_df.copy()
external_predictions_df["label"] = [
    label_mapping[item["label"]] for item in predictions
]

external_predictions_df[["label", "text"]].to_csv(
    prediction_path,
    sep="	",
    header=False,
    index=False,
)
print(f"Saved predictions to: {prediction_path}")


## Optional Hugging Face Hub upload


In [ ]:
# This cell is intentionally disabled so running the full notebook cannot overwrite
# the Hugging Face model repository or expose an access token.
if PUSH_TO_HUB:
    from huggingface_hub import login

    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        raise ValueError("Set the HF_TOKEN environment variable before pushing to the Hub.")

    login(token=hf_token)

    repo_id = "Trotskitten/fake_news_detector"
    model.push_to_hub(
        repo_id,
        commit_message="Upload trained fake-news model"
    )
    tokenizer.push_to_hub(
        repo_id,
        commit_message="Upload tokenizer for trained fake-news model"
    )
else:
    print("Skipping Hugging Face Hub upload. PUSH_TO_HUB is False.")
